# CS 687 · Homework 3 · Project 1, final checkpoint

**Training the model and decoding from it**

> **Before you begin:** Read the repository's `README.md`. It contains the grading and submission rules, including which parts of this notebook you may edit, how to keep the structure that automatic grading depends on, and the required file name of your submission. Read it before changing any cell.

This notebook follows the second part of the Lecture 2 notes. It is meant to run in Google Colab. You may instead run it on your own computer if you can set up the required Python and Jupyter environment yourself. A graphics card is not required, but the training run of section 4 takes a few minutes on any Colab GPU and much longer on the CPU runtime, so choose a GPU runtime under Runtime > Change runtime type before you start.

The homework contains four programming tasks, four short guided calculations and four report responses. Every code answer cell is marked clearly and followed by an immediate check, so you can correct it before continuing. The four report cells are grouped near the end. The final check reruns the guided calculations and the complete public test suite; always run it before submitting your notebook.

The repository ships the complete Homework 1 and Homework 2 code, the tokenizer, the dataset, the input layer, attention, the block and the model, with the course's answers inserted. You do not need your own earlier solutions.

| Notebook section | Notes section |
|---|---|
| 1. Setup | — |
| 2. The text, the tokenizer and the model | The loss on a batch; The two losses |
| 3. Task 1: one step of training | The gradient and backpropagation, recalled; The training loop; One step size for every parameter |
| 4. The run | The training loop; The two losses; Overfitting; Checkpoints; Scaling laws |
| 5. Decoding | Generation as a separate decision; Greedy decoding and its repetition; Temperature, top-p and top-k |
| 6. Run the public tests | — |
| 7. Report | synthesis |
| 8. Finishing the homework | — |


## 1 · Setup
If you are using Colab, the cell below clones the homework repository from GitHub, which holds the code, the text and the tests the notebook uses.


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_NAME = "cs687-hw03-p1-final"
REPO_URL = "https://github.com/LiRA-Lab/cs687-hw03-p1-final.git"
RELEASE = "v1.0"

try:
    import google.colab  # type: ignore
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    repo_root = Path("/content") / REPO_NAME
    if not repo_root.exists():
        subprocess.run(
            ["git", "-c", "advice.detachedHead=false", "clone", "--branch", RELEASE,
             "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
else:
    repo_root = Path.cwd().resolve()
    if repo_root.name == "notebooks":
        repo_root = repo_root.parent
    if not (repo_root / "notebook_support.py").is_file():
        raise RuntimeError(
            "Could not find notebook_support.py. Open this notebook from "
            "somewhere inside the Homework 3 repository."
        )

sys.path.insert(0, str(repo_root))
from notebook_support import prepare_notebook
repo_root = prepare_notebook(repo_root)


## 2 · The text, the tokenizer and the model

Training needs a text, a tokenizer that turns it into ids, windows cut from the stream of ids, and the model. All four come from the earlier homeworks, and this section builds them so that the rest of the notebook has something to train.

The text is a collection of Shakespeare's plays, about a million characters, in `data/tinyshakespeare.txt`. The tokenizer is the byte-pair encoder of Homework 1. Its merge loop is pure Python and its cost grows with the length of the text times the number of merges, so the merge rules are learned on the first 200,000 characters, which takes about a minute, and the whole text is then encoded with those rules in a few seconds. A merge table is an estimate of the statistics of a text, and a sample of the text gives a good one. The vocabulary of the model is every id the tokenizer can produce, $V$, whether or not the text uses it.


In [ ]:
import math
import time
import torch

from cs687 import BPETokenizer

text = open("data/tinyshakespeare.txt", encoding="utf-8").read()
print(f"text: {len(text):,} characters")

started = time.time()
tokenizer = BPETokenizer()
tokenizer.train(text[:200_000], num_merges=500)
ids = tokenizer.encode(text)
print(f"{len(tokenizer.merges)} merges learned in {time.time() - started:.0f} s")

vocab_size = tokenizer.vocab_size
tokens_per_byte = tokenizer.tokens_per_byte(text)
print(f"the text became {len(ids):,} tokens; V = {vocab_size}; {tokens_per_byte:.3f} tokens per byte")
print("the first window of 16 ids:", ids[:16])
print("which spell:", repr(tokenizer.decode(ids[:16])))


### How much text the merges need

Why learn the merges on 200,000 characters and not on the whole text? We measured what the choice costs and what it gives. Each row of the table below learned 500 merges on the first so many characters of the text and then encoded the whole text with that merge table. The time is the time of the learning, on one core of a laptop. The table is given to you as numbers rather than as a cell to run, because running it would take ten minutes and the numbers are what matters.

| characters used to learn the merges | time to learn | tokens the text becomes | tokens per byte |
|---:|---:|---:|---:|
| 25,000 | 3 s | 529,081 | 0.474 |
| 50,000 | 6 s | 524,237 | 0.470 |
| 100,000 | 11 s | 523,397 | 0.469 |
| 200,000 | 23 s | 515,361 | 0.462 |
| 400,000 | 47 s | 504,632 | 0.452 |
| 600,000 | 72 s | 500,238 | 0.448 |
| 800,000 | 99 s | 499,164 | 0.448 |
| 1,000,000 | 127 s | 496,880 | 0.445 |
| 1,115,394, the whole text | 144 s | 496,151 | 0.445 |

The time grows in proportion to the characters, because every merge is a pass over all of them. The compression does not: the merges learned on 25,000 characters already spend only 6 percent more tokens on the text than the merges learned on all of it, and the gain from 200,000 characters to the whole text is 4 percent of the tokens for six times the time. The frequent pieces of the text, the common words, their endings and the space before a word, are in any sample large enough, and the 500 merges are spent on them. A larger sample adds rarer pieces, and each one saves a little. The model of this notebook uses the 200,000-character table.


### How many merges

Why 500 merges? We measured that too, and the answer is not that 500 is best. Each row of the table below learned its merges on the same 200,000 characters and trained the model of this notebook for the same four thousand steps as you will in section 4, with the same seed and the same held-out batches, and measured the held-out loss at the end. The first row has no merges at all, so every byte is a token. The loss in nats per token cannot compare the rows, because each tokenizer cuts the text into different tokens; lecture 1 gave the unit that can, bits per byte, which is the loss in bits per token times the tokens per byte. The time to learn the merges is given relative to the 500 of this notebook, because it grows in proportion to the merges.

| merges | $V$ | tokens per byte | characters in a window of 64 | parameters | time to learn the merges | held-out loss, nats per token | held-out loss, bits per byte |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 0 | 256 | 1.000 | 64 | 667,264 | none | 1.604 | 2.314 |
| 100 | 356 | 0.619 | 103 | 692,864 | 0.2 × | 2.436 | 2.176 |
| 250 | 506 | 0.526 | 122 | 731,264 | 0.5 × | 2.790 | 2.119 |
| 500 | 756 | 0.462 | 139 | 795,264 | 1 × | 3.126 | 2.083 |
| 1,000 | 1,256 | 0.404 | 159 | 923,264 | 1.9 × | 3.399 | 1.979 |
| 2,000 | 2,256 | 0.358 | 179 | 1,179,264 | 3.6 × | 3.982 | 2.057 |

Read down the two loss columns. The loss in nats per token rises with the merges, from 1.6 to 4.0, because a longer token is harder to predict than a single byte. The loss in bits per byte falls, from 2.31 to 1.98, because a window of 64 longer tokens covers more text, 159 characters against 64, and the model learns from more context at every slot. The byte-level model looks best in the first column and is worst in the second, which is the lesson of lecture 1 on our own runs. So more merges are better, up to a point, and the best of the six tables is 1,000 merges, by 0.10 bits per byte over 500.

More merges have a price, and the table shows three parts of it. The time to learn them grows in proportion to the count, because every merge is a pass over the sample: 1,000 merges take twice as long as 500, and 2,000 nearly four times. The model grows with them, because the embedding table and the head hold one row per id: from 500 to 1,000 merges the model gains 128,000 parameters, all of them in those two tables, which go from a quarter of the model to over a third, and at 2,000 merges they are half of it. And each id is trained from fewer occurrences, because the same text becomes fewer tokens spread over more ids: at 500 merges an id occurs about 610 times in the training text on average, at 1,000 about 320, and at 2,000 about 180, so the rows of the rarer ids are barely trained. That last price is what turns the loss back up at 2,000 merges.

The 500 of this notebook is a compromise, not the best choice. It keeps the tokenizer at half the time of 1,000 merges in every run of this notebook, including the one you make before submitting, and it costs 0.10 bits per byte. The lessons of the homework, the two curves, the resume and the decoding rules, are the same at either count. One caveat: the rows compare the same number of steps and not the same amount of text, since a row with more merges sees more characters in its four thousand steps, which favours more merges a little.


### Windows, batches and the held-out text

The previous lecture cut the stream of ids into windows of $T$ slots, each with its targets one slot ahead, and `NextTokenDataset` of Homework 1 does the cutting. The last tenth of the stream is kept out of training and becomes the **held-out text**, from which ten batches are drawn once and kept fixed, so that every measurement of the held-out loss uses the same windows. The window length here is $T = 64$ and a batch holds $B = 32$ windows.


In [ ]:
from cs687 import NextTokenDataset
from cs687.training import sample_batch
from notebook_support import pick_device

CONTEXT = 64
BATCH = 32
device = pick_device()
print("device:", device)

split = int(0.9 * len(ids))
train_dataset = NextTokenDataset(ids[:split], context_len=CONTEXT, stride=32)
held_out_dataset = NextTokenDataset(ids[split:], context_len=CONTEXT, stride=CONTEXT)
print(f"training windows: {len(train_dataset):,}   held-out windows: {len(held_out_dataset):,}")

torch.manual_seed(1)
held_out_batches = [sample_batch(held_out_dataset, BATCH, device) for _ in range(10)]

inputs, targets = sample_batch(train_dataset, BATCH, device)
print("one batch of inputs has shape", tuple(inputs.shape), "and its targets", tuple(targets.shape))
print("window 0, slots 0 to 7:  inputs", inputs[0, :8].tolist())
print("                         targets", targets[0, :8].tolist())


### The model, untrained

The model is the decoder-only transformer of Homework 2, `LanguageModel` in `cs687/model.py`, in the **development configuration** of Project 1. It is small on purpose, so that a run finishes in minutes, and this is what it is made of, from the ids that enter to the scores that leave.

| piece | what it is in this configuration | parameters |
|---|---|---:|
| token table | one vector of width $d_{\mathrm{model}}=128$ for each of the $V=756$ ids | 96,768 |
| position table | one vector of width 128 for each of the $T=64$ slots of a window, which is why the model never reads more than 64 ids at once | 8,192 |
| three blocks | each a layer normalization, causal multi-head attention with $n_{heads}=4$ heads of width $d_{head}=32$ whose four matrices are 128 by 128, a residual connection, a second layer normalization, a feed-forward layer of width 128 to 512 and back with a GELU between, and a second residual connection | 3 × 197,760 |
| final normalization | one more layer normalization on every row | 256 |
| head | the matrix $\mathbf W_u$ of width 128 by 756, which turns each row into one score per id, with no bias | 96,768 |
| total | | 795,264 |

A window of ids enters as a tensor of shape $(B,T)=(32,64)$, becomes a stream of shape $(32,64,128)$ after the two tables, keeps that shape through the three blocks, and leaves as logits of shape $(32,64,756)$. The three blocks hold three quarters of the parameters, and inside a block the feed-forward layer holds two thirds, as Homework 2 counted. Dropout of 0.1 acts in the tables and in every block while the model trains and is off when it is evaluated.

Before training, the parameters are random and the scores at a slot come out nearly equal, and the notes say what the loss is then: the softmax spreads the probability about evenly over the $V$ ids, so the loss at every slot is about $\log V$, the loss of a model that predicts every id with the same probability. The cell below builds the model, prints the count of each kind of parameter, measures the loss on the held-out batches, and prints $\log V$ beside it.


In [ ]:
from cs687 import LanguageModel, count_parameters
from cs687.training import held_out_loss

torch.manual_seed(0)
model = LanguageModel(vocab_size=vocab_size, d_model=128, n_heads=4, n_layers=3, max_len=CONTEXT, dropout=0.1).to(device)
for part, count in count_parameters(model).items():
    print(f"{part:14} {count:>10,}")

untrained_loss = held_out_loss(model, held_out_batches)
print(f"\nloss of the untrained model on the held-out batches: {untrained_loss:.3f} nats per token")
print(f"log V = log {vocab_size} = {math.log(vocab_size):.3f}")


### The starting loss in four units · 4 points

The notes' subsection "The two losses" gives the loss of a model that predicts every id with the same probability, $\log V$ nats per token, and says that the loss is reported in the units the previous lecture gave. Lecture 1 defined those units: the perplexity is the exponential of the loss in nats, the loss in bits per token is the loss in nats divided by $\log 2$, and the loss in bits per byte is the bits per token times the tokens per byte of the tokenizer on the text. Compute the four readings of the starting loss from `vocab_size` and `tokens_per_byte`, which the cells above printed. The check then compares your `start_loss_nats` with `untrained_loss`, the loss the cell above measured on the held-out batches before any training. The two should be close, with the measured one a little above, because the untrained model's scores are not exactly equal. The notes' footnote gives this as the first check of a run: a first loss far above $\log V$ points to a fault in the model, its initialization or the loss calculation, and one far below it to a fault in the loss calculation or to a run that did not start from scratch.

1. `start_loss_nats`: the loss of a model that predicts each of the $V$ ids with the same probability, in nats per token.
2. `start_perplexity`: the same loss read as a perplexity.
3. `start_bits_per_token`: the same loss in bits per token.
4. `start_bits_per_byte`: the same loss in bits per byte, with this tokenizer's tokens per byte on this text.

**Write the four lines in the cell below.**


In [ ]:
# ================= YOUR CODE STARTS HERE =================
raise NotImplementedError("Compute start_loss_nats, start_perplexity, start_bits_per_token and start_bits_per_byte.")
# ================= YOUR CODE ENDS HERE ===================
print(f"nats per token  {start_loss_nats:.3f}")
print(f"perplexity      {start_perplexity:.1f}")
print(f"bits per token  {start_bits_per_token:.3f}")
print(f"bits per byte   {start_bits_per_byte:.3f}")


In [ ]:
from notebook_support import check_starting_loss

check_starting_loss(vocab_size, tokens_per_byte, untrained_loss, start_loss_nats, start_perplexity, start_bits_per_token, start_bits_per_byte)


## 3 · Task 1 · one step of training · 10 points

The notes' subsection "The training loop" gives one step of training as five lines, and none of them adds anything to the mathematics. The forward pass runs the batch through the model and returns the logits, of shape $(B,T,V)$. The loss is the cross-entropy of the previous lecture between the logits and the targets, with the $BT$ slots flattened into one list, so that the loss function sees one row of $V$ scores per scored slot and one target id per row. The backward pass fills the gradient $\mathbf g$ of every parameter. The gradient is clipped, so that its norm is at most the clipping bound and one wild batch can take at most one bounded step. Then the optimizer moves every parameter, and the stored gradients are set to zero, because the framework adds each new gradient to the stored one instead of replacing it.

Write `train_step(model, optimizer, inputs, targets, clip_norm)`. It receives the model in training mode, the optimizer, a batch of inputs and a batch of targets of shape $(B,T)$, and the clipping bound, and it returns the loss on the batch as a plain number. The supplied loop in `cs687/training.py` calls it once per step. The five lines are built from the library pieces below, and this is what each one takes and does.

1. Calling the model on the inputs, as a function, runs the forward pass and returns the logits, a tensor of shape $(B,T,V)$.
2. A tensor's `reshape` method lays the same entries out in a new shape, and a $-1$ in the shape asks it to work out that axis. The loss function wants the logits as a matrix with one row per scored slot and $V$ columns, and the targets as a vector with one id per row, so both are reshaped before the call.
3. `torch.nn.functional.cross_entropy(logits, targets)` takes that matrix and that vector, turns each row of scores into probabilities with the softmax, takes minus the logarithm of the probability given to the row's target, and returns the mean over the rows as a one-element tensor. That is the loss $\mathcal L$ of the notes on the batch.
4. `loss.backward()` runs the backward pass from that tensor and fills the `.grad` field of every parameter the loss depends on with its entry of $\mathbf g$. It adds to a gradient that is already stored instead of replacing it.
5. `torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)` computes the norm of all the stored gradients taken together and, if it exceeds the bound, scales every gradient down in place so that the norm equals the bound.
6. `optimizer.step()` moves every parameter by the optimizer's rule using the stored gradients, and `optimizer.zero_grad()` clears the stored gradients so that the next step starts from nothing.
7. A one-element tensor's `item()` method returns its value as a plain Python number, which is what the loop records.

The task is to put these in the order the notes give, with the clipping between the backward pass and the update.


In [ ]:
import torch


def train_step(model, optimizer, inputs, targets, clip_norm=1.0):
    # ================= YOUR CODE STARTS HERE =================
    raise NotImplementedError("Task 1: implement train_step")
    # ================= YOUR CODE ENDS HERE ===================
print("function defined")


In [ ]:
from notebook_support import run_task1_tests

run_task1_tests(train_step)


### The sizes of the gradients

The notes' subsection "One step size for every parameter" says that the entries of $\mathbf g$ differ in size by orders of magnitude, because the parameters do different jobs: a row of the embedding table gets a gradient only from the slots whose input is its id, a weight of the head touches every slot of every batch, and a weight deep in the stack gets whatever the backward pass carries down to it. The cell below computes the gradient of the untrained model on one batch and prints the typical size of an entry, the root of the mean square, for a few parameters. The optimizer that the run uses, AdamW, divides each entry of the step by the typical size of that entry's gradient, and this is why.


In [ ]:
import torch.nn as nn

model.train()
inputs, targets = sample_batch(train_dataset, BATCH, device)
logits = model(inputs)
loss = nn.functional.cross_entropy(logits.reshape(-1, vocab_size), targets.reshape(-1))
loss.backward()

counts = torch.bincount(inputs.flatten(), minlength=vocab_size)
most_frequent = int(counts.argmax())
absent = int((counts == 0).nonzero()[0])
row_gradient = model.embed.tok.weight.grad

def typical(gradient):
    return gradient.pow(2).mean().sqrt().item()

rows = [
    (f"embedding row of id {most_frequent}, the input of {int(counts[most_frequent])} slots", row_gradient[most_frequent]),
    (f"embedding row of id {absent}, the input of no slot", row_gradient[absent]),
    ("the head W_u, which scores every slot", model.head.weight.grad),
    ("the query matrix of block 0", model.blocks[0].attn.query.weight.grad),
    ("the first feed-forward matrix of block 2", model.blocks[2].ffn.net[0].weight.grad),
]
for label, gradient in rows:
    print(f"{label + ':':<50} {typical(gradient):.2e}")
model.zero_grad(set_to_none=True)


## 4 · The run

The supplied loop `train_run` in `cs687/training.py` is the training loop of the notes. Each pass sets the learning rate for the step from the warmup-and-cosine schedule, draws a batch, runs your `train_step` and records the loss. Every hundred steps it measures the held-out loss on the ten fixed batches and prints both losses, and every five hundred steps it writes the checkpoint with the curve so far. Before the first step it prints $\log V$, the loss of a model that predicts every id with the same probability, which the first printed loss should be near.

The optimizer is the framework's AdamW with two of its settings changed, as is common for language models: the decay rate of the average of the squared gradients is $0.95$ instead of the default $0.999$, and the weight decay is $0.1$ instead of $0.01$. The run has a peak learning rate of $0.003$, a hundred steps of warmup, four thousand steps in all, and the batch and window of section 2. The model trains with dropout, and the optimizer applies weight decay, the two devices the notes name in "The training loop" and "Overfitting". The run takes a few minutes on a Colab GPU and much longer on the CPU runtime. If the checkpoint file already exists, the loop resumes from it through the supplied `load_checkpoint`, so a session that ended in the middle of the run goes on as the same run, and a finished run is restored in seconds when you run the notebook again from the top. Delete `runs/checkpoint.pt` to train from scratch.


In [ ]:
from pathlib import Path

from cs687.training import train_run

STEPS = 4000
PEAK_LEARNING_RATE = 3e-3
WARMUP_STEPS = 100
CHECKPOINT = Path("runs/checkpoint.pt")
CHECKPOINT.parent.mkdir(exist_ok=True)

torch.manual_seed(0)
model = LanguageModel(vocab_size=vocab_size, d_model=128, n_heads=4, n_layers=3, max_len=CONTEXT, dropout=0.1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LEARNING_RATE, betas=(0.9, 0.95), weight_decay=0.1)

curve = train_run(
    model, optimizer, train_dataset, held_out_batches,
    steps=STEPS, batch_size=BATCH, peak_learning_rate=PEAK_LEARNING_RATE, warmup_steps=WARMUP_STEPS,
    checkpoint_path=CHECKPOINT, device=device,
)


### The two curves

The notes' subsection "The two losses" reads three things off this plot: the fast fall at the start and the slow fall after it, the two curves staying close with the held-out loss a little above and smoother, and the parting late in the run, when the training loss keeps falling while the held-out loss flattens and then rises. The lowest point of the held-out loss is the moment at which the model was best. Keep the output of this cell in your saved notebook: the report reads your curves.


In [ ]:
from notebook_support import plot_loss_curves

plot_loss_curves(curve, vocab_size)
best = min(curve, key=lambda point: point["held_out"])
last = curve[-1]
print(f"lowest held-out loss {best['held_out']:.3f} at step {best['step']}, with training loss {best['train']:.3f} there")
print(f"at the last step, {last['step']}: training loss {last['train']:.3f}, held-out loss {last['held_out']:.3f}")


### What dropout and weight decay do

The notes say that dropout and weight decay push the parting of the two curves later. To show what they do, the staff ran the configuration of section 4 three times on a laptop's GPU, with the same seed, the same text and the same held-out batches: as in section 4; with the two devices switched off, no dropout and no weight decay, for the same 4,000 steps; and with the devices on for 20,000 steps, which is 88 passes over the training text. The first is the staff's copy of your run, and yours may come out a little differently, because a run on another graphics card rounds differently.

| step | training loss, as in section 4 | held-out loss, as in section 4 | training loss, without the two devices | held-out loss, without the two devices |
|---:|---:|---:|---:|---:|
| 800 | 3.345 | 3.449 | 3.028 | 3.382 |
| 1,600 | 3.066 | 3.275 | 2.734 | 3.322 |
| 2,400 | 2.888 | 3.194 | 2.440 | 3.410 |
| 3,200 | 2.739 | 3.151 | 2.183 | 3.534 |
| 4,000 | 2.660 | 3.126 | 2.032 | 3.631 |

| step | training loss, the 20,000-step run | held-out loss, the 20,000-step run |
|---:|---:|---:|
| 4,000 | 2.822 | 3.160 |
| 8,000 | 2.694 | 3.149 |
| 12,000 | 2.386 | 3.124 |
| 16,000 | 2.357 | 3.130 |
| 20,000 | 2.474 | 3.153 |

Without the two devices the held-out loss reaches its lowest value, 3.32, at step 1,600 and then climbs, to 3.63 by step 4,000, while the training loss keeps falling to 2.0. That is the parting the notes draw. With the devices on, the held-out loss is 3.13 at step 4,000, and in the 20,000-step run it stays between 3.12 and 3.16 to the end, while the training loss falls on. The two devices do not merely push the parting later; for as long as we ran, they hold the held-out loss down, at the price of a training loss that falls more slowly. The model that is best on new text is the one with the devices on. The 20,000-step run's numbers at step 4,000 differ a little from the 4,000-step run's because its learning rate decays over 20,000 steps and not over 4,000.


### The budget of the run · 4 points

The notes' subsection "Overfitting" says that the training text of a small run is a few hundred thousand tokens and that a run of two thousand steps passes over it several times, and the section "Scaling laws" gives a reference to compare with: the Chinchilla study found a useful balance near twenty training tokens per parameter in its setting, which the notes say is not a requirement for every model or dataset. Count what this run saw and compare it with that reference. Use `STEPS`, `BATCH`, `CONTEXT`, the parameter count of `model`, which `count_parameters` gives as its `"total"` entry, and the number of tokens of the training text, which is `len(train_dataset.inputs)` windows of `CONTEXT` slots at a stride of 32, so about `split` tokens; use `split`.

1. `tokens_seen`: the number of tokens the run processed, the steps times the windows per batch times the slots per window.
2. `passes_over_text`: how many times that is the training text, the tokens seen divided by the tokens of the training text.
3. `reference_tokens`: the tokens that the reference of twenty per parameter gives for a model of this size.
4. `factor_below_reference`: by what factor this run falls below the reference, the tokens of the reference divided by the tokens seen.

**Write the four lines in the cell below.**


In [ ]:
parameter_count = count_parameters(model)["total"]
text_tokens = split
# ================= YOUR CODE STARTS HERE =================
raise NotImplementedError("Compute tokens_seen, passes_over_text, reference_tokens and factor_below_reference.")
# ================= YOUR CODE ENDS HERE ===================
print(f"tokens seen       {tokens_seen:>14,}")
print(f"passes over text  {passes_over_text:>14.1f}")
print(f"reference tokens  {reference_tokens:>14,}")
print(f"factor below ref. {factor_below_reference:>14.1f}")


In [ ]:
from notebook_support import check_run_budget

check_run_budget(STEPS, BATCH, CONTEXT, parameter_count, text_tokens, tokens_seen, passes_over_text, reference_tokens, factor_below_reference)


### A resumed run is the same run

The three-step standard of the notes, on your own run. The first cell continues the run for three steps from where the loop left it, which is the state the last checkpoint holds. Then a fresh model and optimizer, built from a different seed on purpose, load the checkpoint through the supplied `load_checkpoint` in `cs687/training.py`, which is the notes' algorithm "resume a run from a checkpoint", and train three steps. The two lists of losses must agree to the last digit. The last part loads only the parameters and the generator into a third fresh model, gives its optimizer the run's last learning rate so that only the optimizer's running averages are missing, and trains three steps: the losses after the first step differ, and the report asks why.


In [ ]:
from cs687.training import load_checkpoint, set_generator_state


def three_steps(model, optimizer):
    model.train()
    losses = []
    for _ in range(3):
        inputs, targets = sample_batch(train_dataset, BATCH, device)
        losses.append(train_step(model, optimizer, inputs, targets, 1.0))
    return losses

def fresh_model_and_optimizer(seed):
    torch.manual_seed(seed)
    fresh = LanguageModel(vocab_size=vocab_size, d_model=128, n_heads=4, n_layers=3, max_len=CONTEXT, dropout=0.1).to(device)
    return fresh, torch.optim.AdamW(fresh.parameters(), lr=PEAK_LEARNING_RATE, betas=(0.9, 0.95), weight_decay=0.1)

continued = three_steps(model, optimizer)

resumed_model, resumed_optimizer = fresh_model_and_optimizer(seed=123)
next_step, extra = load_checkpoint(CHECKPOINT, resumed_model, resumed_optimizer)
resumed = three_steps(resumed_model, resumed_optimizer)

print("the run continued:     ", [round(loss, 6) for loss in continued])
print("a fresh model resumed: ", [round(loss, 6) for loss in resumed])
print("largest difference:    ", max(abs(a - b) for a, b in zip(continued, resumed)))
print(f"the file said to continue from step {next_step}, and its curve has {len(extra['curve'])} points")

# The same, with the optimizer's state left out: the parameters and the generator come back, and the
# optimizer gets the run's last learning rate, so that only its running averages are missing.
partial_model, partial_optimizer = fresh_model_and_optimizer(seed=123)
saved = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
partial_model.load_state_dict(saved["model"])
for group in partial_optimizer.param_groups:
    group["lr"] = saved["optimizer"]["param_groups"][0]["lr"]
set_generator_state(saved["rng"])
without_optimizer = three_steps(partial_model, partial_optimizer)
print("\nwithout the optimizer's state:", [round(loss, 6) for loss in without_optimizer])
print("largest difference from the run:", max(abs(a - b) for a, b in zip(continued, without_optimizer)))


## 5 · Decoding

Training ends with a model that gives, for any prefix, a distribution over the next id, and producing text is a loop around it with a rule inside: how to pick one id from the distribution. The notes show the three rules on one distribution of eight ids, and the cell below holds that table as a tensor of logits, with the text of each token in place of its id. Every number of the next two calculations and of the three tasks is computed from it.


In [ ]:
LOGITS = torch.tensor([3.2, 2.1, 1.8, 0.9, 0.4, 0.1, -0.6, -1.2])
TOKENS = ["the", "cat", "dog", "house", "idea", "running", "purple", "seven"]

probabilities = torch.softmax(LOGITS, dim=-1)
print(f"{'token':>8} {'logit':>6} {'probability':>12}")
for token, logit, probability in zip(TOKENS, LOGITS.tolist(), probabilities.tolist()):
    print(f"{token:>8} {logit:>6.1f} {probability:>12.3f}")


### Temperature on the table · 4 points

The notes' subsection "Temperature, top-p and top-k" divides the logits by a positive number $\tau$, the temperature, before the softmax, so that
$$p_v=\frac{\exp(z_v/\tau)}{\sum_{u\in\mathcal V}\exp(z_u/\tau)} .$$
A temperature below one sharpens the distribution toward its favourite and one above one flattens it toward uniform. Compute, from `LOGITS`, the probability of the favourite, the first id, and the mass of the tail, the last four ids, at two of the temperatures the notes use, $\tau=0.5$ and $\tau=1.5$. `torch.softmax` with `dim=-1` turns a vector of logits into probabilities. Once the check passes, it prints the whole table at $\tau=0.5$, $1$ and $1.5$, with the mass of cat and dog, the two plausible alternatives, and of the last four ids; keep that output, because the report reads it.

1. `favourite_at_half`: the probability of "the" at $\tau=0.5$.
2. `favourite_at_three_halves`: the probability of "the" at $\tau=1.5$.
3. `tail_at_half`: the mass of the last four ids at $\tau=0.5$.
4. `tail_at_three_halves`: the mass of the last four ids at $\tau=1.5$.

**Write the four lines in the cell below.**


In [ ]:
# ================= YOUR CODE STARTS HERE =================
raise NotImplementedError("Compute favourite_at_half, favourite_at_three_halves, tail_at_half and tail_at_three_halves.")
# ================= YOUR CODE ENDS HERE ===================
print(f"tau = 0.5: favourite {float(favourite_at_half):.3f}, tail {float(tail_at_half):.3f}")
print(f"tau = 1.5: favourite {float(favourite_at_three_halves):.3f}, tail {float(tail_at_three_halves):.3f}")


In [ ]:
from notebook_support import check_temperature_table, show_temperatures

check_temperature_table(LOGITS, favourite_at_half, favourite_at_three_halves, tail_at_half, tail_at_three_halves)
show_temperatures(LOGITS, TOKENS)


### The two cuts on the table · 4 points

The same subsection cuts the tail two ways. Top-p takes the ids in order of probability from the most probable down until the mass they carry together reaches $p$, keeps that set, and scales the kept probabilities up to sum to one. Top-k keeps the $k$ most probable ids whatever mass they carry and scales them up in the same way. Compute, from `probabilities` of the table, which is already in order from the most probable down:

1. `cumulative_mass`: the running sum of the probabilities, one entry per id, so that entry $i$ is the mass of the first $i+1$ ids. The `cumsum` method of a tensor gives running sums along an axis.
2. `kept_at_p_09`: the number of ids top-p keeps at $p=0.9$, counting the id that carries the sum past $0.9$.
3. `favourite_under_top_p`: the probability of "the" after the cut at $p=0.9$ and the renormalization, which is its probability divided by the mass of the kept set.
4. `favourite_under_top_k`: the probability of "the" after top-k at $k=3$ and the renormalization.

**Write the four lines in the cell below.**


In [ ]:
# ================= YOUR CODE STARTS HERE =================
raise NotImplementedError("Compute cumulative_mass, kept_at_p_09, favourite_under_top_p and favourite_under_top_k.")
# ================= YOUR CODE ENDS HERE ===================
print("cumulative mass:", [round(value, 3) for value in cumulative_mass.tolist()])
print(f"ids kept at p = 0.9: {kept_at_p_09}")
print(f"the favourite under top-p at 0.9: {float(favourite_under_top_p):.3f}")
print(f"the favourite under top-k at 3:   {float(favourite_under_top_k):.3f}")


In [ ]:
from notebook_support import check_two_cuts

check_two_cuts(LOGITS, cumulative_mass, kept_at_p_09, favourite_under_top_p, favourite_under_top_k)


### Task 2 · temperature · 3 points

Write `apply_temperature(logits, temperature)`: divide the logits by the temperature and return the result, a tensor of the same shape. The sampler handles greedy decoding on its own, so a temperature of zero or below is not meaningful here; raise a `ValueError` for it. The tests check the function against the numbers of the table.


In [ ]:
def apply_temperature(logits, temperature):
    # ================= YOUR CODE STARTS HERE =================
    raise NotImplementedError("Task 2: implement apply_temperature")
    # ================= YOUR CODE ENDS HERE ===================
print("function defined")


In [ ]:
from notebook_support import run_task2_tests

run_task2_tests(apply_temperature)


### Task 3 · top-p · 7 points

Write `top_p_filter(logits, p)`. The notes' rule: take the ids from the most probable down until their mass reaches $p$, keep that set, and remove every other id. In code the cut is made on the logits, by setting the logit of every removed id to minus infinity, which is the device of the causal mask: the softmax then gives those ids exactly zero and renormalizes the survivors on its own. The function returns a tensor of the same shape in which every kept id keeps its own logit, unchanged, and not a probability or a log-probability; when $p$ is at least one it returns the logits unchanged. The last axis is the vocabulary, and any leading axes are batches.

One detail decides the edge case the notes name. The condition that removes an id is that the mass before it already reaches $p$, that is, is at least $p$, and not that the mass including it does, so that the favourite is kept even when it alone carries more than $p$. Written the other way round, the rule keeps nothing. The ids are taken in the order the sort gives them, so when ids of equal probability meet at the cut, the sort decides which of them is kept, and the set holds exactly as many ids as the running mass gives. The docstring of the stub in `cs687/sampling.py` names the library functions: `torch.sort` with `descending=True` returns the sorted values and the index each came from, the `cumsum` method gives running sums, a boolean tensor's `scatter` along the last axis with those indices carries a mask over the sorted order back to the original order, and `masked_fill` replaces the entries where a mask is true.


In [ ]:
def top_p_filter(logits, p):
    # ================= YOUR CODE STARTS HERE =================
    raise NotImplementedError("Task 3: implement top_p_filter")
    # ================= YOUR CODE ENDS HERE ===================
print("function defined")


In [ ]:
from notebook_support import run_task3_tests

run_task3_tests(top_p_filter)


### Task 4 · top-k · 4 points

Write `top_k_filter(logits, k)`. The notes' rule: keep the $k$ most probable ids whatever mass they carry, and remove the rest by setting their logits to minus infinity, as in top-p, so that every kept id keeps its own logit, unchanged. An id whose logit equals the $k$-th largest is kept too, so when ids tie at the $k$-th place, more than $k$ survive. Return the logits unchanged when $k$ is zero or at least the vocabulary size. `torch.topk` returns the $k$ largest values of each row and their indices in descending order, so the smallest kept value of a row is the last of them, and `masked_fill` replaces the entries where a mask is true.


In [ ]:
def top_k_filter(logits, k):
    # ================= YOUR CODE STARTS HERE =================
    raise NotImplementedError("Task 4: implement top_k_filter")
    # ================= YOUR CODE ENDS HERE ===================
print("function defined")


In [ ]:
from notebook_support import run_task4_tests

run_task4_tests(top_k_filter)


### The same prompt under five rules

The supplied `generate` in `cs687/sampling.py` is the loop of the notes: it runs the prefix through the model, reads the distribution at the last slot, picks an id by the rule, appends it and runs the model again. The supplied `sample_next` applies your three rules in the notes' order, temperature first and then the cut. The cell first restores the model from the checkpoint, since the three-step demonstration moved it, and then continues the prompt "The king" a hundred tokens under five rules from the same seed. Nothing changes between the samples except the rule. Keep the output: the report reads your samples.


In [ ]:
from cs687.sampling import generate

load_checkpoint(CHECKPOINT, model, optimizer)
model.eval()

prompt = "The king"
prompt_ids = torch.tensor([tokenizer.encode(prompt)], device=device)

rules = [
    ("greedy", dict(greedy=True)),
    ("sampling, temperature 1", dict(temperature=1.0)),
    ("temperature 0.5", dict(temperature=0.5)),
    ("top-k 10", dict(top_k=10)),
    ("top-p 0.9", dict(top_p=0.9)),
]
for name, settings in rules:
    torch.manual_seed(0)
    continuation = generate(model, prompt_ids, max_new_tokens=100, **settings)
    print(f"--- {name} " + "-" * (50 - len(name)))
    print(tokenizer.decode(continuation[0].tolist()))
    print()


### Before and after training

The closing of the notes places a sample from the untrained model beside one from the trained one. The cell builds a fresh model of the same configuration from the same seed the run started from, which is the model the run began with, and continues the same prompt by plain sampling, as Homework 2 did. The architecture is identical, and everything between the two samples is the loop of section 4.


In [ ]:
torch.manual_seed(0)
untrained = LanguageModel(vocab_size=vocab_size, d_model=128, n_heads=4, n_layers=3, max_len=CONTEXT, dropout=0.1).to(device)
untrained.eval()

torch.manual_seed(0)
before = generate(untrained, prompt_ids, max_new_tokens=40, temperature=1.0)
torch.manual_seed(0)
after = generate(model, prompt_ids, max_new_tokens=40, temperature=1.0)

print("before training:", repr(tokenizer.decode(before[0].tolist())))
print()
print("after training: ", repr(tokenizer.decode(after[0].tolist())))


## 6 · Run the public tests

The immediate checks above give feedback after each guided calculation and programming task. This final cell repeats all four calculation checks and runs all 30 public tests against the four task implementations currently loaded in the notebook. Always run it before submitting your notebook. It does not require you to copy anything into the repository's `.py` files. Staff grading uses a separate, staff-controlled test suite.


In [ ]:
from notebook_support import (
    check_run_budget,
    check_starting_loss,
    check_temperature_table,
    check_two_cuts,
    run_all_public_tests,
)

check_starting_loss(vocab_size, tokens_per_byte, untrained_loss, start_loss_nats, start_perplexity, start_bits_per_token, start_bits_per_byte)
check_run_budget(STEPS, BATCH, CONTEXT, parameter_count, text_tokens, tokens_seen, passes_over_text, reference_tokens, factor_below_reference)
check_temperature_table(LOGITS, favourite_at_half, favourite_at_three_halves, tail_at_half, tail_at_three_halves)
check_two_cuts(LOGITS, cumulative_mass, kept_at_p_09, favourite_under_top_p, favourite_under_top_k)

run_all_public_tests(train_step, apply_temperature, top_p_filter, top_k_filter)


## 7 · Report

Use the four cells below to explain what the notebook shows. Do not paste whole output tables or repeat the code line by line. Where a question asks for values, take them from your own outputs or from the table the question names. These report cells are part of the notebook submission, and so are the outputs of sections 4 and 5 that they rest on.


### 1 · Reading the run · 15 points

In section 4 you trained the development configuration for four thousand steps with dropout and weight decay and plotted its two losses against the step. Then you read the staff's table of the same configuration without the two devices and of a run of 20,000 steps. The notes' subsections "The two losses" and "Overfitting" explain what you saw.

- How does the gap between your training loss and your held-out loss change over your run, and how does that differ from the parting the notes describe?
- Which of the two losses tells you whether the model is learning the language, and why that one? What can lower the other one without the model getting better on new text, and what in this run makes that possible?
- Assume the staff's run saved a checkpoint at each step in its table. In the staff's run without dropout and weight decay, which checkpoint would you keep, and why not the last one, although its training loss is the lowest? What is the model doing in the steps after the one you keep?

Write approximately 100–200 words.

**Response:**

_Replace this line with your response._


### 2 · Resuming a run · 15 points

In section 4 you ran the notes' three-step standard on your own run. A fresh model and optimizer that loaded the checkpoint trained three steps with exactly the run's losses, and a third model that was restored with the run's last learning rate but without the optimizer's state did not. The notes' subsection "Checkpoints" explains both.

- The notes list four things the file must hold. Name them, and for each say what would go wrong in the steps after a resume if it were missing.
- The model restored without the optimizer's state: what is that state, and why did its absence change the losses after the first step?

Write approximately 100–200 words.

**Response:**

_Replace this line with your response._


### 3 · Greedy decoding and sampling · 15 points

In section 5 you continued the prompt "The king" from your own trained model under greedy decoding and under plain sampling at temperature one, from the same seed, and nothing changed between the two samples except the rule. The notes' subsections "Generation as a separate decision" and "Greedy decoding and its repetition" explain what you saw.

- Quote a phrase that repeats in your greedy sample. Explain how the phrase, once it is in the prefix, makes itself likely again, and why a rule that always takes the most probable id cannot leave the loop.
- Your greedy and sampled continuations came from the same model, prompt and seed. What was different in how they were produced, and why does that show the model is not broken?
- What is the weakness of plain sampling at temperature one? Answer in terms of the tail of the distribution: what the many improbable ids hold together although each is tiny, how often a draw therefore lands among them, and what one such id does to the text after it.

Write approximately 100–200 words.

**Response:**

_Replace this line with your response._


### 4 · Top-p against top-k · 15 points

In section 5 you computed, on the notes' table of eight ids, the probabilities after two temperatures and after the two cuts. The notes' subsection "Temperature, top-p and top-k" explains the three rules.

- On the table, where do top-k at $k=3$ and top-p at $p=0.9$ cut, and why do they keep different numbers of ids?
- What is the weakness of a fixed count $k$, and how does a mass avoid it? Describe what $k=3$ does when the model is nearly certain, and when twenty continuations are plausible.
- Using the table printed after your temperature calculation, explain why temperature cannot replace a cut: what happens to the two plausible alternatives, cat and dog, and to the tail of the last four ids when $\tau$ rises, and when it falls.

Write approximately 100–200 words.

**Response:**

_Replace this line with your response._


## 8 · Finishing the homework

Submit only this notebook. Before uploading it to Moodle, rename it to `homework03_colab_STUDENTNUMBER.ipynb`, replacing `STUDENTNUMBER` with your student number. For example: `homework03_colab_22206543.ipynb`.

Before submitting, restart the Colab runtime and run the notebook from top to bottom. The run of section 4 is restored from its checkpoint in seconds if the session's disk is still there, and takes a few minutes on a GPU runtime otherwise. A run that trained again can come out a little differently, so then check that every value and phrase your report quotes matches the new outputs. Confirm that the final check reports that all four guided calculations and all 30 public tests passed, and make sure none of the four report cells still contains its placeholder. Save the notebook in Colab with its outputs, rename it, download it with File > Download > Download .ipynb, and upload that exact file, and only that file, to the Moodle assignment for Homework 3.

The outputs of sections 4 and 5, the printed losses, the plot of the two curves and the samples, are the evidence your report is read against, so keep them in the saved notebook. Grading runs your notebook again from a clean environment to check the calculations and the four tasks, with a short training run in place of yours. The report explanations are the part that matters most. You may write the code with the help of a coding agent, but the quiz and the examination ask related questions without assistance, so make sure you can explain whatever you submit.
